In [0]:
WITH CREDIT_ASSESSMENT AS
(
SELECT  _id, `_created_by.Name`, 
        `_created_at.value`,  
        `_created_at.timezone`, 
        `_submitted_at.value`, 
        `_modified_by.Name`, 
        `_modified_at.value`,
        array_join(array(regexp_replace(split_part(split_part(_current_assigned_to, ',', 2), ':', 2), '"', ''),
        regexp_replace(split_part(split_part(_current_assigned_to, ',', 5), ':', 2), '"', ''),
        regexp_replace(split_part(split_part(_current_assigned_to, ',', 8), ':', 2), '"', ''),
        regexp_replace(split_part(split_part(_current_assigned_to, ',',11), ':', 2), '"', '')), ', ', '') AS _current_assigned_to, 
        _current_step, 
        `_completed_at.value`,  
        _status, _progress, Name, 
        CASE WHEN Requested_credit_limit <= 100000 THEN 'LOCAL'
        WHEN Requested_credit_limit > 100000 AND Requested_credit_limit <= 1000000 THEN 'REGIONAL'
        WHEN Requested_credit_limit > 1000000 THEN 'GLOBAL' ELSE NULL END AS DOA_By_Requested_CL,
        TRIM(split_part(Name, '|', -2 )) AS No_Credit_Assessment_KF,
        Group_Name__Hid,
        TRIM(CASE WHEN regexp_instr(Group_Name__Hid, '[0-9]') = '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, 1, regexp_instr(group_name__hid, '[A-Z]') -1), '') 
        WHEN regexp_instr(Group_Name__Hid, '[0-9]') > '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, regexp_instr(Group_Name__Hid, '[0-9]') -1, len(Group_Name__Hid)), '') 
        ELSE Group_Name__Hid END) AS Group_Name__Hid_Cleaned,                        -- SE REMUEVEN CÓDIGOS DE COLUMNA Group_Name__Hid 
        TRIM(CASE WHEN SUBSTRING(Group_Name__Hid_Cleaned, -1) IN ('-', '/', '|') 
        THEN REPLACE(Group_Name__Hid_Cleaned, SUBSTRING(Group_Name__Hid_Cleaned, -1), '') 
        ELSE Group_Name__Hid_Cleaned END) AS Group_Name__Hid_Cleaned_,               -- SE REMUEVEN CARACTERES ESPECIALES DE COLUMNA ANTERIOR
        TRIM(regexp_replace(split_part(split_part(Customer_Number, ',', 1), ':', 2), '[-"A-Z]', '')) AS Customer_Number,
        Business_Segment_1, Region, CountryEntity, Type_of_request, Payment_Terms, Updated_Payment_Terms, 
        cast(Current_credit_limit AS double) AS Current_credit_limit,
        cast(Requested_credit_limit AS double) AS Requested_credit_limit, 
        cast(Key_Money_Amount AS double) AS Key_Money_Amount, 
        cast(Credit_Limit_Total_Hidden AS double) AS Credit_Limit_Total_Hidden,
        Select_Security_Type, Total_Security_Offered, Security_Type_Hidden, Projected_Volume, Projected_Gross_Margin_USD, Payout_Years, TAT_calculation, Apprehensions, 
        date_format(Last_Reviewed_Date,'yyyy-MM-dd') AS Last_Reviewed_Date, 
        Type_of_review, Risk_Profile, Financial_Strength, Financial_statement_analysis, Flow_ID_for_Link

FROM kissflow.default.databricks_output_2025_08_29 
WHERE `_created_at.value` >= '2025-01-01' 
),

NAV_CUSTOMER AS 
(SELECT Name,
        UIDName,
        ROW_NUMBER() OVER(PARTITION BY Name ORDER BY Name DESC) AS RN
FROM prodsqldwh.adls.nav_customer_dim
WHERE CompanyCode IN ('COS', 'EPA', 'ELS', 'EBE', 'ENC', 'PEG', 'EGU', 'HON', 'HOR', 'EHO', 'PCA', 'PCP')
AND IsDeleted = False
AND UIDName <> 'null'
),

CA_DIGITAL_WORKSPACE AS
(
SELECT CA.* EXCEPT(CA.Group_Name__Hid, 
                   CA.Group_Name__Hid_Cleaned),
       CASE WHEN NC.UIDName IS NULL THEN CA.Group_Name__Hid_Cleaned_ ELSE NC.UIDName END AS UIDName -- SE ASIGNA EL MISMO NOMBRE PARA LOS CASOS QUE 
FROM CREDIT_ASSESSMENT CA                                                                             -- NO SE ENCONTRÓ EL UID Name
LEFT JOIN NAV_CUSTOMER NC
 ON CA.Group_Name__Hid_Cleaned_ = NC.Name
WHERE NC.RN = 1 OR NC.RN IS NULL 
)

SELECT *
FROM CA_DIGITAL_WORKSPACE
ORDER BY `_created_at.value` DESC;



